In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
pip install unidecode python-calamine -q

In [11]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import monotonically_increasing_id, split, trim, col, Column, coalesce, isnull, translate, round
from pyspark.sql.functions import avg, col, rand, when, lower, regexp_extract, regexp_replace, expr, create_map, lit
import pyspark.sql.functions as F
import pandas as pd
import geopandas as gpd
import unicodedata
from unidecode import unidecode
import matplotlib as plt
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import glob
from numpy import quantile
from sklearn.preprocessing import MinMaxScaler
import numpy as np
import requests, io

In [12]:
acentos = "áàãâäéèêëíìîïóòõôöúùûüçÁÀÃÂÄÉÈÊËÍÌÎÏÓÒÕÔÖÚÙÛÜÇ"
sem_acentos = "aaaaaeeeeiiiiooooouuuucAAAAAEEEEIIIIOOOOOUUUUC"

In [13]:
spark = SparkSession.builder \
    .appName("Trata base de dados criminais") \
    .getOrCreate()

In [14]:
# https://prefeitura.sp.gov.br/web/subprefeituras/w/munic%C3%ADpio-de-s%C3%A3o-paulo-subprefeituras-e-distritos-municipais-1
# https://drive.prefeitura.sp.gov.br/cidade/secretarias/subprefeituras/upload/pinheiros/11_regioes.pdf
df_regioes = (pd.read_excel("drive/MyDrive/Colab Notebooks/Analise_Urbana_SP/BASES/distritos_bairros/regioes.xlsx")
                .assign(ds_nome=lambda df: df["Bairro"].str.upper().str.translate(str.maketrans(acentos, sem_acentos))))

In [ ]:
path = "https://www.ssp.sp.gov.br/assets/estatistica/transparencia/spDados/SPDadosCriminais_2025.xlsx"

df_SPDadosCriminais = spark.createDataFrame(pd.concat([pd.read_excel(path, sheet_name="JAN_A_JUN_2025", engine="calamine"), 
                                                       pd.read_excel(path, sheet_name="JUL_A_DEZ_2025", engine="calamine")], 
                                                ignore_index=True).astype(str))


In [ ]:
dados_categoria = [
    ("APREENSÃO DE ENTORPECENTES", "DROGAS"),
    ("ROUBO DE CARGA", "PATRIMONIAL"),
    ("HOMICÍDIO CULPOSO POR ACIDENTE DE TRÂNSITO", "TRÂNSITO"),
    ("HOMICÍDIO DOLOSO", "LETAL INTENCIONAL"),
    ("TENTATIVA DE HOMICÍDIO", "LETAL INTENCIONAL"),
    ("LATROCÍNIO", "LETAL INTENCIONAL"),
    ("LESÃO CORPORAL DOLOSA", "VIOLÊNCIA PESSOAL"),
    ("LESÃO CORPORAL CULPOSA - OUTRAS", "VIOLÊNCIA PESSOAL"),
    ("LESÃO CORPORAL SEGUIDA DE MORTE", "VIOLÊNCIA PESSOAL"),
    ("ROUBO - OUTROS", "PATRIMONIAL"),
    ("ROUBO DE VEÍCULO", "PATRIMONIAL"),
    ("FURTO - OUTROS", "PATRIMONIAL"),
    ("FURTO DE VEÍCULO", "PATRIMONIAL"),
    ("ESTUPRO TOTAL", "VIOLÊNCIA SEXUAL"),
    ("TRÁFICO DE ENTORPECENTES", "DROGAS"),
    ("PORTE DE ENTORPECENTES", "DROGAS"),
    ("PORTE DE ARMA", "ARMAS"),
    ("HOMICÍDIO CULPOSO OUTROS", "LETAL NÃO INTENCIONAL"),
    ("HOMICÍDIO DOLOSO POR ACIDENTE DE TRÂNSITO", "TRÂNSITO"),
    ("EXTORSÃO MEDIANTE SEQUESTRO", "VIOLÊNCIA GRAVE"),
    ("EXTORSÃO", "PATRIMONIAL"),
    ("SEQUESTRO E CÁRCERE PRIVADO", "VIOLÊNCIA GRAVE"),
    ("AMEAÇA", "VIOLÊNCIA PESSOAL"),
    ("VIAS DE FATO", "VIOLÊNCIA PESSOAL"),
    ("ESTELIONATO", "PATRIMONIAL"),
    ("RECEPTAÇÃO", "PATRIMONIAL"),
    ("DANO", "PATRIMONIAL"),
    ("INVASÃO DE DOMICÍLIO", "VIOLÊNCIA PESSOAL"),
    ("VIOLÊNCIA DOMÉSTICA", "VIOLÊNCIA PESSOAL"),
    ("ASSÉDIO SEXUAL", "VIOLÊNCIA SEXUAL"),
    ("IMPORTUNAÇÃO SEXUAL", "VIOLÊNCIA SEXUAL"),
    ("ATO OBSCENO", "VIOLÊNCIA SEXUAL"),
    ("CORRUPÇÃO DE MENORES", "VIOLÊNCIA SEXUAL"),
    ("MAUS TRATOS", "VIOLÊNCIA PESSOAL"),
    ("ABANDONO DE INCAPAZ", "VIOLÊNCIA PESSOAL"),
    ("OMISSÃO DE SOCORRO", "VIOLÊNCIA PESSOAL"),
    ("DESACATO", "OUTROS"),
    ("DESOBEDIÊNCIA", "OUTROS"),
    ("RESISTÊNCIA", "OUTROS"),
    ("USO DE DOCUMENTO FALSO", "OUTROS"),
    ("FALSIDADE IDEOLÓGICA", "OUTROS"),
    ("MOEDA FALSA", "OUTROS"),
    ("CRIME AMBIENTAL", "OUTROS"),
    ("CRIME ELEITORAL", "OUTROS"),
    ("CRIME MILITAR", "OUTROS"),
    ("OUTROS NÃO CRIMINAL", "OUTROS"),
    ("COMUNICAÇÃO DE ÓBITO", "OUTROS"),
    ("MORTE SUSPEITA", "OUTROS"),
    ("ENCONTRO DE CADÁVER", "OUTROS"),
    ("DESAPARECIMENTO DE PESSOA", "OUTROS"),
    ("LOCALIZAÇÃO DE VEÍCULO", "OUTROS"),
    ("LOCALIZAÇÃO DE PESSOA", "OUTROS"),
    ("APREENSÃO DE VEÍCULO", "OUTROS"),
    ("FURTO DE CARGA", "PATRIMONIAL"),
    ("LESÃO CORPORAL CULPOSA POR ACIDENTE DE TRÂNSITO", "TRÂNSITO"),
    ("RECUPERAÇÃO DE VEÍCULO", "OUTROS")]

df_categoria = spark.createDataFrame(dados_categoria, ["NATUREZA_APURADA", "Categoria_Crime"])

In [ ]:
df_dados_criminais = (df_SPDadosCriminais.withColumn("LATITUDE",  F.trim(F.col("LATITUDE")))
                                         .withColumn("LONGITUDE", F.trim(F.col("LONGITUDE")))
                                         .withColumn("LATITUDE",  F.regexp_replace("LATITUDE",  ",", "."))
                                         .withColumn("LONGITUDE", F.regexp_replace("LONGITUDE", ",", "."))
                                         .withColumn("LATITUDE",  F.regexp_replace(F.col("LATITUDE"),  r"\.+$", ""))
                                         .withColumn("LONGITUDE", F.regexp_replace(F.col("LONGITUDE"), r"\.+$", ""))
                                         .filter(F.col("LATITUDE").isNotNull() &
                                                 F.col("LONGITUDE").isNotNull() &
                                                 (F.col("LATITUDE")  != "") &
                                                 (F.col("LONGITUDE") != "") &
                                                 (F.col("LATITUDE")  != "0") &
                                                 (F.col("LONGITUDE") != "0") &
                                                 (F.col("LATITUDE")  != "0.0") &
                                                 (F.col("LONGITUDE") != "0.0") &
                                                 (~F.upper(F.col("LATITUDE")).isin("NAN", "NULL", "NUL.L")) &
                                                 (~F.upper(F.col("LONGITUDE")).isin("NAN", "NULL", "NUL+L")))
                                         .withColumn("LATITUDE",  F.expr("try_cast(LATITUDE as double)"))
                                         .withColumn("LONGITUDE", F.expr("try_cast(LONGITUDE as double)"))
                                         .filter((col("COD IBGE") == "3550308") &  
                                                 F.col("LATITUDE").isNotNull() & 
                                                 F.col("LONGITUDE").isNotNull()  & 
                                                 F.col("LATITUDE").between(-24.0, -23.2) & 
                                                 F.col("LONGITUDE").between(-47.2, -46.2))
                                         .join(df_categoria, on="NATUREZA_APURADA", how="left"))


In [ ]:
df_SPDadosCriminais.write.mode("overwrite").parquet("drive/MyDrive/Colab Notebooks/Analise_Urbana_SP/BASES/dados_criminais/df_SPDadosCriminais.parquet")
df_dados_criminais.write.mode("overwrite").parquet("drive/MyDrive/Colab Notebooks/Analise_Urbana_SP/BASES/dados_criminais/df_dados_criminais.parquet")


+----------------+-----------------+--------------------+--------------------+--------------+------+------+-------------+------------------+------------------+------------+--------------------+--------------------+--------------------+--------------------+-----------------+-----------------+-----------------+----------------------------+-------------------------------+----------------------------+----------------------------+----------------+--------------------+---------------+---------------+-------+-------+-----------+--------+---------------+
|NATUREZA_APURADA|NOME_DEPARTAMENTO|      NOME_SECCIONAL|      NOME_DELEGACIA|NOME_MUNICIPIO|NUM_BO|ANO_BO|DATA_REGISTRO|DATA_OCORRENCIA_BO|HORA_OCORRENCIA_BO|DESC_PERIODO|     DESCR_TIPOLOCAL|  DESCR_SUBTIPOLOCAL|              BAIRRO|          LOGRADOURO|NUMERO_LOGRADOURO|         LATITUDE|        LONGITUDE|NOME_DELEGACIA_CIRCUNSCRICAO|NOME_DEPARTAMENTO_CIRCUNSCRICAO|NOME_SECCIONAL_CIRCUNSCRICAO|NOME_MUNICIPIO_CIRCUNSCRICAO|         RUBRICA| 

Aqui iremos criar um "depara" de bairro por latitude e longitude, visto que não existe um padrão entre os nomes de bairros na base de dados criminais e outras bases

In [ ]:
gdf_distritos = gpd.read_file("https://raw.githubusercontent.com/codigourbano/distritos-sp/master/distritos-sp.geojson")

gdf_distritos["centroid"] = gdf_distritos.geometry.centroid
gdf_distritos["centroid_x"] = gdf_distritos["centroid"].x
gdf_distritos["centroid_y"] = gdf_distritos["centroid"].y

df_coords_distintas_pd = (df_dados_criminais.select("LATITUDE", "LONGITUDE").dropDuplicates().toPandas())

gdf_coords = gpd.GeoDataFrame(df_coords_distintas_pd, geometry=gpd.points_from_xy(df_coords_distintas_pd["LONGITUDE"], 
                                                                                  df_coords_distintas_pd["LATITUDE"]), crs="EPSG:4326")


/tmp/ipykernel_96819/2549409506.py:3: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf_distritos["centroid"] = gdf_distritos.geometry.centroid


In [ ]:
gdf_coords.head()

AttributeError: 'GeoDataFrame' object has no attribute 'show'

In [ ]:
df_coords_bairro = spark.createDataFrame(((gpd.sjoin(gdf_coords, gdf_distritos[["geometry", "ds_nome"]],
                                                    how="left", predicate="within")
                                                   .dropna(subset=["ds_nome"])
                                                   .drop_duplicates())
                                                   .merge(df_regioes, how="left", on="ds_nome")))

PySparkTypeError: [CANNOT_INFER_TYPE_FOR_FIELD] Unable to infer the type of the field `geometry`.

In [ ]:
# A perda aqui é minima (menos de 1% dos dados) logo, não vale a pena tentar classificar, fora que, 
# boa parte dos casos, se tratam de registros na grande São Paulo, por isso eu troquei left por inner

# left
# +-------------------+-----------------+
# |            % nulos|      % não nulos|
# +-------------------+-----------------+
# |0.06312566742242119|99.93687433257757|
# +-------------------+-----------------+

# inner 
# +-------+-----------+
# |% nulos|% não nulos|
# +-------+-----------+
# |    0.0|      100.0|
# +-------+-----------+

df_crimes_agg_bairro = (df_dados_criminais.join(F.broadcast(df_coords_bairro), on=["LATITUDE", "LONGITUDE"], how="inner")
                                          .agg(F.countDistinct("NUM_BO").alias("QTD_CRIMES"))
                                          .orderBy("Categoria_Crime"))

{"ts": "2026-04-29 19:05:14.884", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `Categoria_Crime` cannot be resolved. Did you mean one of the following? [`QTD_CRIMES`]. SQLSTATE: 42703", "context": {"file": "java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o303.sort.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `Categoria_Crime` cannot be resolved. Did you mean one of the following? [`QTD_CRIMES`]. SQLSTATE: 42703;\n'Sort ['Categoria_Crime ASC NULLS FIRST], true\n+- Aggregate [count(distinct NUM_BO#4) AS QTD_CRIMES#169L]\n   +- Project [LATITUDE#38, LONGITUDE#39, NATUREZA_APURADA#23, NOME_DEPARTAMENTO#0

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `Categoria_Crime` cannot be resolved. Did you mean one of the following? [`QTD_CRIMES`]. SQLSTATE: 42703;
'Sort ['Categoria_Crime ASC NULLS FIRST], true
+- Aggregate [count(distinct NUM_BO#4) AS QTD_CRIMES#169L]
   +- Project [LATITUDE#38, LONGITUDE#39, NATUREZA_APURADA#23, NOME_DEPARTAMENTO#0, NOME_SECCIONAL#1, NOME_DELEGACIA#2, NOME_MUNICIPIO#3, NUM_BO#4, ANO_BO#5, DATA_REGISTRO#6, DATA_OCORRENCIA_BO#7, HORA_OCORRENCIA_BO#8, DESC_PERIODO#9, DESCR_TIPOLOCAL#10, DESCR_SUBTIPOLOCAL#11, BAIRRO#12, LOGRADOURO#13, NUMERO_LOGRADOURO#14, NOME_DELEGACIA_CIRCUNSCRICAO#17, NOME_DEPARTAMENTO_CIRCUNSCRICAO#18, NOME_SECCIONAL_CIRCUNSCRICAO#19, NOME_MUNICIPIO_CIRCUNSCRICAO#20, RUBRICA#21, DESCR_CONDUTA#22, MES_ESTATISTICA#24, ... 10 more fields]
      +- Join Inner, ((LATITUDE#38 = LATITUDE#41) AND (LONGITUDE#39 = LONGITUDE#42))
         :- Project [NATUREZA_APURADA#23, NOME_DEPARTAMENTO#0, NO

In [ ]:
df_coords_bairro.show()

In [ ]:
df_crimes_agg_bairro.show()

In [ ]:
# df_crimes_agg_bairro.write.mode("overwrite").parquet("drive/MyDrive/Colab Notebooks/Analise_Urbana_SP/BASES/dados_criminais/dados_criminais.parquet")


